
## Folder 09 — File 04: Incremental Gold Refresh:


- Gold tables contain business-level aggregations, so refreshing them is different from merging individual Silver records.
- For example, when an order changes, its daily sales total, channel sales, product performance, customer lifetime value and executive KPIs may all be affected.
- This notebook orchestrates your existing Folder 07 Gold transformation notebooks in dependency order. 
- It gives you a repeatable scheduled refresh, but the existing Gold notebooks use full-table overwrites. 
- That is a full Gold recomputation, not a true partition-level incremental refresh. 
- I won't label a full overwrite as incremental just because it runs on a schedule.

In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime
import traceback


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

BASE_PATH = (
    "/Volumes/workspace/default/"
    "data_volume/ecommerce_lakehouse"
)


GOLD_NOTEBOOK_ROOT = (
    "/Workspace/Data-Enginnering/Data-Engineering-Python/E-Commerce-Lakehouse-Project/07_gold_business_analytics"
)

REPORT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "gold_refresh_execution_report"
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)


# ============================================================
# 3. SILVER SOURCE TABLES
# ============================================================

SILVER_SOURCE_TABLES = [

    "silver_customers",
    "silver_products",
    "silver_orders",
    "silver_order_items",
    "silver_payments",
    "silver_shipments",
    "silver_inventory"

]


# ============================================================
# 4. GOLD NOTEBOOK EXECUTION ORDER
# ============================================================

# These are the existing Folder 07 transformation notebooks.
# They must be available in the Databricks Workspace.

GOLD_REFRESH_TASKS = [

    {
        "task_name": "daily_sales",
        "notebook": "01_gold_daily_sales",
        "description": "Daily and channel sales aggregation"
    },

    {
        "task_name": "product_performance",
        "notebook": "02_gold_product_performance",
        "description": "Product and category performance"
    },

    {
        "task_name": "customer_lifetime_value",
        "notebook": "03_gold_customer_lifetime_value",
        "description": "Customer lifetime value and segments"
    },

    {
        "task_name": "payment_metrics",
        "notebook": "04_gold_payment_metrics",
        "description": "Payment and reconciliation metrics"
    },

    {
        "task_name": "shipping_metrics",
        "notebook": "05_gold_shipping_metrics",
        "description": "Shipping and delivery analytics"
    },

    {
        "task_name": "inventory_status",
        "notebook": "06_gold_inventory_status",
        "description": "Inventory and warehouse analytics"
    },

    {
        "task_name": "executive_kpis",
        "notebook": "07_gold_executive_kpis",
        "description": "Executive-level business KPIs"
    }

]


# ============================================================
# 5. EXPECTED GOLD TABLES
# ============================================================

EXPECTED_GOLD_TABLES = [

    "gold_daily_sales",
    "gold_channel_daily_sales",

    "gold_product_performance",
    "gold_category_performance",

    "gold_customer_lifetime_value",
    "gold_customer_segments",

    "gold_payment_metrics",
    "gold_payment_method_performance",
    "gold_order_payment_summary",

    "gold_shipping_metrics",

    "gold_inventory_status",
    "gold_warehouse_inventory",
    "gold_inventory_snapshot_trends",
    "gold_inventory_alerts",

    "gold_executive_kpis"

]


# ============================================================
# 6. HELPER: CHECK TABLE EXISTENCE
# ============================================================

def table_exists(table_name):

    return spark.catalog.tableExists(table_name)


# ============================================================
# 7. HELPER: VALIDATE SILVER INPUTS
# ============================================================

def validate_silver_sources():

    print("\nVALIDATING SILVER SOURCE TABLES")

    missing_tables = []

    for table_name in SILVER_SOURCE_TABLES:

        full_table_name = (
            f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
        )

        if table_exists(full_table_name):

            print(f"AVAILABLE: {full_table_name}")

        else:

            print(f"MISSING: {full_table_name}")

            missing_tables.append(full_table_name)

    if missing_tables:

        raise ValueError(
            "Missing required Silver tables: "
            + ", ".join(missing_tables)
        )

    print("All required Silver tables are available.")


# ============================================================
# 8. HELPER: BUILD NOTEBOOK PATH
# ============================================================

def build_notebook_path(notebook_name):

    return (
        f"{GOLD_NOTEBOOK_ROOT}/{notebook_name}"
    )


# ============================================================
# 9. HELPER: EXECUTE GOLD NOTEBOOK
# ============================================================

def execute_gold_notebook(task):

    task_name = task["task_name"]

    notebook_name = task["notebook"]

    notebook_path = build_notebook_path(
        notebook_name
    )

    print("\n" + "=" * 75)

    print(
        f"STARTING GOLD REFRESH: {task_name}"
    )

    print(
        f"NOTEBOOK PATH: {notebook_path}"
    )

    print(
        f"DESCRIPTION: {task['description']}"
    )

    print("=" * 75)

    start_time = datetime.now()

    try:

        # Execute the Gold notebook.
        # timeout_seconds = 0 means no notebook timeout.

        notebook_output = dbutils.notebook.run(
            notebook_path,
            0,
            {}
        )

        end_time = datetime.now()

        duration_seconds = (
            end_time - start_time
        ).total_seconds()

        result = {

            "task_name": task_name,

            "notebook_name": notebook_name,

            "notebook_path": notebook_path,

            "execution_status": "SUCCESS",

            "start_timestamp": start_time,

            "end_timestamp": end_time,

            "duration_seconds": float(duration_seconds),

            "execution_message": str(
                notebook_output
            )[:1000]

        }

        print(
            f"SUCCESS: {task_name}"
        )

        print(
            f"Duration: {duration_seconds:.2f} seconds"
        )

        return result

    except Exception as error:

        end_time = datetime.now()

        duration_seconds = (
            end_time - start_time
        ).total_seconds()

        result = {

            "task_name": task_name,

            "notebook_name": notebook_name,

            "notebook_path": notebook_path,

            "execution_status": "FAILED",

            "start_timestamp": start_time,

            "end_timestamp": end_time,

            "duration_seconds": float(duration_seconds),

            "execution_message": str(error)[:1000]

        }

        print(
            f"FAILED: {task_name}"
        )

        print(str(error))

        return result


# ============================================================
# 10. HELPER: VALIDATE GOLD OUTPUT TABLES
# ============================================================

def validate_gold_outputs():

    print("\nVALIDATING GOLD OUTPUT TABLES")

    output_results = []

    missing_tables = []

    for table_name in EXPECTED_GOLD_TABLES:

        full_table_name = (
            f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
        )

        if table_exists(full_table_name):

            row_count = spark.table(
                full_table_name
            ).count()

            status = (
                "AVAILABLE"
                if row_count > 0
                else "EMPTY"
            )

            print(
                f"{full_table_name}: "
                f"{row_count} rows ({status})"
            )

            output_results.append({

                "table_name": full_table_name,

                "row_count": int(row_count),

                "validation_status": status

            })

            if row_count == 0:

                missing_tables.append(
                    f"{full_table_name} is empty"
                )

        else:

            print(
                f"MISSING: {full_table_name}"
            )

            output_results.append({

                "table_name": full_table_name,

                "row_count": 0,

                "validation_status": "MISSING"

            })

            missing_tables.append(
                f"{full_table_name} does not exist"
            )

    return output_results, missing_tables


# ============================================================
# 11. CREATE REFRESH REPORT TABLE
# ============================================================

if not table_exists(REPORT_TABLE):

    report_schema = """

        task_name STRING,
        notebook_name STRING,
        notebook_path STRING,
        execution_status STRING,
        start_timestamp TIMESTAMP,
        end_timestamp TIMESTAMP,
        duration_seconds DOUBLE,
        execution_message STRING,
        report_timestamp TIMESTAMP

    """

    spark.createDataFrame(
        [],
        report_schema
    ).write.format(
        "delta"
    ).saveAsTable(
        REPORT_TABLE
    )


# ============================================================
# 12. START GOLD REFRESH PIPELINE
# ============================================================

pipeline_start_time = datetime.now()

execution_results = []

pipeline_failed = False

try:

    # --------------------------------------------------------
    # 12.1 VALIDATE SILVER INPUTS
    # --------------------------------------------------------

    validate_silver_sources()

    # --------------------------------------------------------
    # 12.2 EXECUTE GOLD NOTEBOOKS IN ORDER
    # --------------------------------------------------------

    for task in GOLD_REFRESH_TASKS:

        result = execute_gold_notebook(task)

        execution_results.append(result)

        # Stop dependent refreshes if one notebook fails.
        if result["execution_status"] == "FAILED":

            pipeline_failed = True

            print(
                "Stopping Gold refresh because "
                "a required task failed."
            )

            break

    # --------------------------------------------------------
    # 12.3 VALIDATE GOLD OUTPUTS
    # --------------------------------------------------------

    if not pipeline_failed:

        output_results, missing_outputs = (
            validate_gold_outputs()
        )

        if missing_outputs:

            pipeline_failed = True

            print(
                "Gold output validation failed:"
            )

            for issue in missing_outputs:

                print(issue)

except Exception as error:

    pipeline_failed = True

    execution_results.append({

        "task_name": "pipeline_validation",

        "notebook_name": "N/A",

        "notebook_path": "N/A",

        "execution_status": "FAILED",

        "start_timestamp": pipeline_start_time,

        "end_timestamp": datetime.now(),

        "duration_seconds": 0.0,

        "execution_message": str(error)[:1000]

    })


# ============================================================
# 13. SAVE EXECUTION REPORT
# ============================================================

if execution_results:

    report_df = spark.createDataFrame(
        execution_results
    )

    report_df = report_df.withColumn(
        "report_timestamp",
        F.current_timestamp()
    )

    (

        report_df

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(REPORT_TABLE)

    )

    print("\nGOLD REFRESH EXECUTION REPORT")

    display(
        report_df.orderBy("start_timestamp")
    )


# ============================================================
# 14. FINAL PIPELINE STATUS
# ============================================================

pipeline_end_time = datetime.now()

total_duration = (
    pipeline_end_time - pipeline_start_time
).total_seconds()

print("\n" + "=" * 75)

print("GOLD REFRESH PIPELINE SUMMARY")

print("=" * 75)

print(
    f"Start time: {pipeline_start_time}"
)

print(
    f"End time: {pipeline_end_time}"
)

print(
    f"Duration: {total_duration:.2f} seconds"
)

print(
    f"Successful tasks: "
    f"{sum(1 for r in execution_results if r['execution_status'] == 'SUCCESS')}"
)

print(
    f"Failed tasks: "
    f"{sum(1 for r in execution_results if r['execution_status'] == 'FAILED')}"
)

if pipeline_failed:

    print("FINAL STATUS: FAILED")

    raise RuntimeError(
        "Gold refresh pipeline failed. "
        f"Check {REPORT_TABLE}."
    )

else:

    print("FINAL STATUS: SUCCESS")

print(
    f"Execution report: {REPORT_TABLE}"
)


In [0]:
%sql
SELECT *
FROM workspace.ecommerce_gold.gold_refresh_execution_report
ORDER BY report_timestamp DESC;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_refresh_execution_report;

In [0]:
%sql
SELECT
    task_name,
    notebook_name,
    execution_status,
    start_timestamp,
    end_timestamp,
    duration_seconds,
    execution_message,
    report_timestamp
FROM workspace.ecommerce_gold.gold_refresh_execution_report
WHERE report_timestamp = (
    SELECT MAX(report_timestamp)
    FROM workspace.ecommerce_gold.gold_refresh_execution_report
)
ORDER BY start_timestamp;

In [0]:
# ============================================================
# GOLD TABLE VALIDATION
# ============================================================

EXPECTED_GOLD_TABLES = [
    "gold_daily_sales",
    "gold_channel_daily_sales",
    "gold_product_performance",
    "gold_category_performance",
    "gold_customer_lifetime_value",
    "gold_customer_segments",
    "gold_payment_metrics",
    "gold_payment_method_performance",
    "gold_order_payment_reconciliation",
    "gold_shipping_metrics",
    "gold_inventory_status",
    "gold_warehouse_inventory",
    "gold_inventory_snapshot_trends",
    "gold_inventory_alerts",
    "gold_executive_kpis"
]

print("=" * 75)
print("GOLD TABLE VALIDATION")
print("=" * 75)

successful_tables = 0
empty_tables = 0
failed_tables = 0

for table in EXPECTED_GOLD_TABLES:

    full_name = f"workspace.ecommerce_gold.{table}"

    try:
        count = spark.table(full_name).count()

        if count > 0:
            successful_tables += 1
            print(f"PASS  | {table} | {count} rows")

        else:
            empty_tables += 1
            print(f"EMPTY | {table} | 0 rows")

    except Exception as e:
        failed_tables += 1
        print(f"FAIL  | {table} | {str(e)[:300]}")

print("\n" + "=" * 75)
print("VALIDATION SUMMARY")
print("=" * 75)

print(f"Expected tables : {len(EXPECTED_GOLD_TABLES)}")
print(f"Successful      : {successful_tables}")
print(f"Empty           : {empty_tables}")
print(f"Failed          : {failed_tables}")

if successful_tables == len(EXPECTED_GOLD_TABLES):
    print("\nALL GOLD TABLES VALIDATED")
else:
    print("\nGOLD TABLE VALIDATION NEEDS ATTENTION")

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_gold
LIKE '*payment*';